## RAG evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph/blob/main/notebooks/en/7_6_RAG_evaluation_en.ipynb)

| | |
|---|---|
| 🎯 **What you'll learn** | How to check whether a RAG app really works: retrieval metrics (Recall@K, Precision@K, MRR), LLM-as-judge for faithfulness and correctness, citation accuracy, latency and token usage on a small golden dataset. |
| 🔑 **API keys** | `OPENAI_API_KEY` |
| 💲 **API cost** | low – about 50 short requests |
| ⏱️ **Run time** | ~2 min |

### Install libraries

In [1]:
# Install this notebook's direct dependencies.
# Skipped when pip is missing, e.g. in the `uv sync` environment, which already has them (see README).
import importlib.util
if importlib.util.find_spec("pip"):
    %pip install -qU langchain-openai pandas python-dotenv
# Google Colab: add OPENAI_API_KEY in the Secrets panel (🔑) and grant this notebook access.
# Uncomment these lines to load it from Colab Secrets into this runtime:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Locally, put OPENAI_API_KEY=... in a .env file; do not commit the key.

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

### What do we measure?

A RAG app can fail in two places: the **retriever** may not find the right fragments, or the **LLM** may answer badly even with the right context. That is why we measure both stages separately, plus the cost of the whole pipeline.

| Stage | Metric | What do we check? |
|---|---|---|
| Retrieval | **Recall@K** | Were the needed fragments found in the top K? |
| Retrieval | **Precision@K** | How many of the top K fragments are relevant? |
| Retrieval | **MRR** | How high was the first correct result? |
| Generation | **Faithfulness** | Does the answer follow from the provided sources (no hallucinations)? |
| Generation | **Answer correctness** | Is the answer correct compared with the reference? |
| Generation | **Citation accuracy** | Were the right sources cited? |
| Cost | **Latency** | How long did it take to generate the answer? |
| Cost | **Token usage** | How many resources did the app consume? |

You don't need thousands of examples to start: **10–20 questions with reference answers and the IDs of the right documents** are enough to catch most regressions.

### 1. Knowledge base with document IDs
Stable IDs are the key to evaluation – without them we can't say *which* document should have been found. Here each article is a single chunk; with splitting, give every chunk the ID of its source document (e.g. in `metadata`).

In [3]:
from langchain_core.documents import Document

# Help-center articles of a fictional SaaS product "Orbitly".
# Each article has a stable ID - the evaluation dataset refers to these IDs.
KNOWLEDGE_BASE = {
    "plans-01": "Orbitly has three plans. Free: up to 3 projects and 1 GB of storage. Pro: $12 per user per month, unlimited projects and 100 GB of storage. Enterprise: custom pricing, SSO and a dedicated account manager.",
    "billing-01": "Invoices are issued on the 1st day of every month and sent to the billing email address. Pro customers pay by credit card. Enterprise customers can also pay by bank transfer with 30-day payment terms.",
    "billing-02": "Annual plans can be refunded within 14 days of purchase. Monthly plans are non-refundable, but cancelling a monthly plan stops the next charge.",
    "trial-01": "The Pro trial lasts 21 days and does not require a credit card. When the trial ends, the workspace is automatically downgraded to the Free plan and data above the Free limits becomes read-only.",
    "security-01": "All data is encrypted at rest with AES-256 and in transit with TLS 1.3. Data is stored either in Frankfurt (EU) or in Virginia (US); the region is chosen when the workspace is created.",
    "security-02": "Two-factor authentication (2FA) works with an authenticator app or a hardware security key (WebAuthn). Admins on Pro and Enterprise plans can require 2FA for all workspace members.",
    "sso-01": "Single sign-on (SSO) with SAML 2.0 and OIDC is available only on the Enterprise plan. Supported identity providers are Okta, Microsoft Entra ID (Azure AD) and Google Workspace.",
    "api-01": "The REST API allows 600 requests per minute per workspace on Pro and 3000 on Enterprise. When the limit is exceeded, the API returns HTTP 429 with a Retry-After header.",
    "api-02": "API keys are created in Settings -> Developer. A key is shown only once, can be scoped as read-only or read-write, and expires after 365 days.",
    "export-01": "A single project can be exported to JSON, CSV or PDF from Project -> Export. Admins can request a full workspace export as a ZIP file; it is ready within 24 hours and the download link is valid for 7 days.",
    "support-01": "Free users get help on the community forum. Pro users get email support with a first response within 24 business hours. Enterprise users get 24/7 phone support and a 1-hour response time for critical incidents.",
    "retention-01": "Deleted projects stay in the Trash for 30 days and are then permanently removed. Backups are kept for 35 days. After an account is closed, all data is deleted within 60 days.",
    "integrations-01": "Orbitly integrates with Slack, GitHub, Google Drive and Jira. Webhooks are available on Pro and Enterprise; failed deliveries are retried up to 5 times with exponential backoff.",
    "mobile-01": "Mobile apps are available for iOS 16+ and Android 10+. In offline mode changes are synced when the connection returns; conflicting edits are resolved by keeping both versions.",
}

docs = [Document(page_content=text, metadata={"id": doc_id}) for doc_id, text in KNOWLEDGE_BASE.items()]
print(f"Documents in the knowledge base: {len(docs)}")

Documents in the knowledge base: 14


### 2. Golden evaluation dataset
15 questions written the way real users ask them (not copied from the documents). Three of them need **two** documents – these test whether the retriever can gather complete context.

In [4]:
# Golden dataset: question, reference answer, IDs of the documents that contain the answer.
EVAL_DATASET = [
    {"question": "How much does the Pro plan cost?",
     "reference": "$12 per user per month.",
     "relevant_ids": ["plans-01"]},
    {"question": "Can I get a refund for a monthly plan?",
     "reference": "No, monthly plans are non-refundable; cancelling only stops the next charge.",
     "relevant_ids": ["billing-02"]},
    {"question": "How long is the free trial and do I need a credit card?",
     "reference": "The Pro trial lasts 21 days and no credit card is required.",
     "relevant_ids": ["trial-01"]},
    {"question": "What happens to my workspace when the trial ends?",
     "reference": "It is downgraded to the Free plan (3 projects, 1 GB); data above the Free limits becomes read-only.",
     "relevant_ids": ["trial-01", "plans-01"]},
    {"question": "Which identity providers can I use for SSO?",
     "reference": "Okta, Microsoft Entra ID (Azure AD) and Google Workspace, via SAML 2.0 or OIDC, on the Enterprise plan only.",
     "relevant_ids": ["sso-01"]},
    {"question": "What happens if my integration sends too many API requests on the Pro plan?",
     "reference": "Above 600 requests per minute the API returns HTTP 429 with a Retry-After header.",
     "relevant_ids": ["api-01"]},
    {"question": "Where is my data stored and how is it protected?",
     "reference": "In Frankfurt (EU) or Virginia (US), chosen at workspace creation; encrypted with AES-256 at rest and TLS 1.3 in transit.",
     "relevant_ids": ["security-01"]},
    {"question": "Can an admin force every member to use 2FA?",
     "reference": "Yes, admins on Pro and Enterprise can require 2FA (authenticator app or hardware key).",
     "relevant_ids": ["security-02"]},
    {"question": "How do I download all data from my workspace?",
     "reference": "An admin requests a full workspace export as a ZIP; it is ready within 24 hours and the link is valid for 7 days.",
     "relevant_ids": ["export-01"]},
    {"question": "How quickly does support respond to Pro customers?",
     "reference": "By email, with a first response within 24 business hours.",
     "relevant_ids": ["support-01"]},
    {"question": "I deleted a project two weeks ago. Can I still restore it?",
     "reference": "Yes, deleted projects stay in the Trash for 30 days.",
     "relevant_ids": ["retention-01"]},
    {"question": "Where do I create an API key and how long is it valid?",
     "reference": "In Settings -> Developer; the key expires after 365 days.",
     "relevant_ids": ["api-02"]},
    {"question": "What happens when my webhook endpoint is temporarily down?",
     "reference": "Failed deliveries are retried up to 5 times with exponential backoff.",
     "relevant_ids": ["integrations-01"]},
    {"question": "Can I pay by bank transfer?",
     "reference": "Only on the Enterprise plan, with 30-day payment terms; Pro is paid by credit card.",
     "relevant_ids": ["billing-01"]},
    {"question": "We need SSO and 24/7 phone support. Which plan should we buy?",
     "reference": "Enterprise - it is the only plan with SSO and 24/7 phone support.",
     "relevant_ids": ["sso-01", "support-01"]},
]
print(f"Questions in the evaluation set: {len(EVAL_DATASET)}")

Questions in the evaluation set: 15


### 3. Retrieval metrics: Recall@K, Precision@K, MRR
These are plain functions on lists of IDs – no LLM needed, so they are fast, cheap and deterministic.

In [5]:
def recall_at_k(retrieved: list[str], relevant: list[str], k: int) -> float:
    """Share of the relevant documents that appear in the top-k results."""
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant)


def precision_at_k(retrieved: list[str], relevant: list[str], k: int) -> float:
    """Share of the top-k results that are relevant."""
    return len(set(retrieved[:k]) & set(relevant)) / k


def reciprocal_rank(retrieved: list[str], relevant: list[str]) -> float:
    """1 / position of the first relevant result (0 if none was found). MRR = mean over questions."""
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            return 1 / rank
    return 0.0


# quick self-check on a toy example
retrieved, relevant = ["a", "b", "c"], ["b", "d"]
assert recall_at_k(retrieved, relevant, k=3) == 0.5      # found b, missed d
assert precision_at_k(retrieved, relevant, k=3) == 1 / 3  # 1 of 3 results is relevant
assert reciprocal_rank(retrieved, relevant) == 0.5       # first hit at position 2
print("Metrics OK")

Metrics OK


#### Vector index
For a small dataset an in-memory store is enough; the metrics work the same with FAISS, Chroma or pgvector.

In [6]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embeddings)


def retrieve_ids(question: str, k: int) -> list[str]:
    return [d.metadata["id"] for d in vectorstore.similarity_search(question, k=k)]


print(retrieve_ids("Is my data encrypted?", k=3))

['security-01', 'security-02', 'mobile-01']


#### Evaluating the retriever for different K
Notice the trade-off: as K grows, **recall rises** (more chances to find the document) but **precision falls** (more noise in the context, more tokens and a higher risk of distracting the model).

In [7]:
import pandas as pd

# Retrieval costs no LLM calls, so we fetch the top 5 once and slice it for every k.
retrieval_rows = []
for item in EVAL_DATASET:
    top5 = retrieve_ids(item["question"], k=5)
    row = {"question": item["question"], "relevant": item["relevant_ids"], "retrieved": top5,
           "MRR": reciprocal_rank(top5, item["relevant_ids"])}
    for k in (1, 3, 5):
        row[f"Recall@{k}"] = recall_at_k(top5, item["relevant_ids"], k)
        row[f"Precision@{k}"] = precision_at_k(top5, item["relevant_ids"], k)
    retrieval_rows.append(row)

retrieval_df = pd.DataFrame(retrieval_rows)
metric_cols = ["Recall@1", "Recall@3", "Recall@5", "Precision@1", "Precision@3", "Precision@5", "MRR"]
retrieval_df[metric_cols].mean().round(3).to_frame("mean")

,mean
Recall@1,0.867
Recall@3,0.967
Recall@5,0.967
Precision@1,0.933
Precision@3,0.356
Precision@5,0.213
MRR,0.967


In [8]:
# Questions where the first hit was not at position 1 - worth looking at by hand
retrieval_df.loc[retrieval_df["MRR"] < 1, ["question", "relevant", "retrieved", "MRR"]]

,question,relevant,retrieved,MRR
0,How much does the Pro plan cost?,[plans-01],"[trial-01, plans-01, support-01, billing-02, b...",0.5


### 4. RAG pipeline with citations, latency and token usage
The model returns a structured answer (`with_structured_output`) with a list of source IDs, so citation accuracy can be checked automatically. `get_usage_metadata_callback` sums the tokens of all chat-model calls in the block, and `time.perf_counter` measures latency.

In [9]:
import time

from langchain_core.callbacks import get_usage_metadata_callback
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

TOP_K = 3


class CitedAnswer(BaseModel):
    """Answer to the user's question with the IDs of the documents it is based on."""
    answer: str = Field(description="Answer based only on the context. Say you don't know if the context has no answer.")
    sources: list[str] = Field(description="IDs of the context documents that support the answer, e.g. ['plans-01'].")


rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are the Orbitly help-center assistant. Answer concisely using ONLY the context. "
               "Every context document starts with its ID in square brackets - cite the IDs you used.\n\n"
               "CONTEXT:\n{context}"),
    ("user", "{question}"),
])
rag_chain = rag_prompt | ChatOpenAI(model="gpt-6-luna").with_structured_output(CitedAnswer)


def format_docs(documents: list[Document]) -> str:
    return "\n\n".join(f"[{d.metadata['id']}] {d.page_content}" for d in documents)


def ask(question: str, k: int = TOP_K) -> dict:
    """Run the RAG pipeline and measure latency and token usage."""
    start = time.perf_counter()
    with get_usage_metadata_callback() as usage:  # collects usage_metadata of every chat-model call inside
        context_docs = vectorstore.similarity_search(question, k=k)
        result = rag_chain.invoke({"context": format_docs(context_docs), "question": question})
    return {
        "answer": result.answer,
        "cited": result.sources,
        "retrieved": [d.metadata["id"] for d in context_docs],
        "context": format_docs(context_docs),
        "latency_s": time.perf_counter() - start,
        "tokens": sum(u["total_tokens"] for u in usage.usage_metadata.values()),
    }


ask("Can I pay by bank transfer?")

{'answer': 'Enterprise customers can pay by bank transfer with 30-day payment terms. Pro customers pay by credit card.',
 'cited': ['billing-01'],
 'retrieved': ['billing-01', 'billing-02', 'trial-01'],
 'context': '[billing-01] Invoices are issued on the 1st day of every month and sent to the billing email address. Pro customers pay by credit card. Enterprise customers can also pay by bank transfer with 30-day payment terms.\n\n[billing-02] Annual plans can be refunded within 14 days of purchase. Monthly plans are non-refundable, but cancelling a monthly plan stops the next charge.\n\n[trial-01] The Pro trial lasts 21 days and does not require a credit card. When the trial ends, the workspace is automatically downgraded to the Free plan and data above the Free limits becomes read-only.',
 'latency_s': 1.6510707950219512,
 'tokens': 334}

### 5. LLM-as-judge: faithfulness and answer correctness
- **Faithfulness** compares the answer with the *retrieved context* – it detects hallucinations, even if the answer happens to be true.
- **Answer correctness** compares the answer with the *reference* – it detects wrong or incomplete answers, even if they are faithful to a bad context.

The judge returns its reasoning *before* the verdict, which makes the grade more reliable and easier to audit. Use a model at least as strong as the one being evaluated.

In [10]:
class Verdict(BaseModel):
    reasoning: str = Field(description="Short justification, written before the verdict.")
    passed: bool = Field(description="Final verdict.")


judge = ChatOpenAI(model="gpt-6.1-terra").with_structured_output(Verdict)

faithfulness_prompt = ChatPromptTemplate.from_messages([
    ("system", "You check RAG answers for hallucinations. The answer passes if EVERY factual claim in it "
               "is supported by the context. An answer saying it does not know also passes."),
    ("user", "CONTEXT:\n{context}\n\nANSWER:\n{answer}"),
])

correctness_prompt = ChatPromptTemplate.from_messages([
    ("system", "You compare an answer with a reference answer. The answer passes if it contains the key facts "
               "of the reference and does not contradict it. Extra correct details are fine."),
    ("user", "QUESTION:\n{question}\n\nREFERENCE:\n{reference}\n\nANSWER:\n{answer}"),
])

faithfulness_judge = faithfulness_prompt | judge
correctness_judge = correctness_prompt | judge

# sanity check: the judge must catch an invented fact
faithfulness_judge.invoke({
    "context": "[plans-01] Pro: $12 per user per month.",
    "answer": "Pro costs $12 per user per month and includes a free domain.",
})

Verdict(reasoning='The price is supported, but the context does not mention a free domain.', passed=False)

### 6. Citation accuracy
We compare the IDs cited by the model with the IDs from the dataset – again a plain function, no LLM.

In [11]:
def citation_scores(cited: list[str], relevant: list[str]) -> tuple[float, float]:
    """Citation precision (are the cited sources right?) and recall (were all needed sources cited?)."""
    hits = len(set(cited) & set(relevant))
    precision = hits / len(set(cited)) if cited else 0.0
    return precision, hits / len(relevant)


assert citation_scores(["a", "b"], ["a"]) == (0.5, 1.0)
assert citation_scores([], ["a"]) == (0.0, 0.0)

### 7. Full evaluation run

In [12]:
def evaluate_example(item: dict, k: int = TOP_K) -> dict:
    out = ask(item["question"], k=k)
    faithful = faithfulness_judge.invoke({"context": out["context"], "answer": out["answer"]})
    correct = correctness_judge.invoke({"question": item["question"], "reference": item["reference"],
                                        "answer": out["answer"]})
    cit_precision, cit_recall = citation_scores(out["cited"], item["relevant_ids"])
    return {
        "question": item["question"],
        "answer": out["answer"],
        "cited": out["cited"],
        "relevant": item["relevant_ids"],
        f"Recall@{k}": recall_at_k(out["retrieved"], item["relevant_ids"], k),
        f"Precision@{k}": precision_at_k(out["retrieved"], item["relevant_ids"], k),
        "MRR": reciprocal_rank(out["retrieved"], item["relevant_ids"]),
        "faithfulness": faithful.passed,
        "correctness": correct.passed,
        "citation_precision": cit_precision,
        "citation_recall": cit_recall,
        "latency_s": out["latency_s"],  # RAG pipeline only, judges are not counted
        "tokens": out["tokens"],
        "judge_notes": f"F: {faithful.reasoning} | C: {correct.reasoning}",
    }


results = pd.DataFrame([evaluate_example(item) for item in EVAL_DATASET])
results[["question", "answer", "cited", "faithfulness", "correctness", "latency_s", "tokens"]]

,question,answer,cited,faithfulness,correctness,latency_s,tokens
0,How much does the Pro plan cost?,The Pro plan costs $12 per user per month.,[plans-01],True,True,1.204422,349
1,Can I get a refund for a monthly plan?,Monthly plans are non-refundable. You can canc...,[billing-02],True,True,2.069316,331
2,How long is the free trial and do I need a cre...,"The Pro free trial lasts 21 days, and you don’...",[trial-01],True,True,1.437269,338
3,What happens to my workspace when the trial ends?,"When your 21-day Pro trial ends, your workspac...",[trial-01],True,False,1.470875,365
4,Which identity providers can I use for SSO?,"Orbitly SSO supports Okta, Microsoft Entra ID ...",[sso-01],True,False,1.349598,380
5,What happens if my integration sends too many ...,"On the Pro plan, the API limit is 600 requests...",[api-01],True,True,1.119021,368
6,Where is my data stored and how is it protected?,Your data is stored in either Frankfurt (EU) o...,[security-01],True,True,1.434601,371
7,Can an admin force every member to use 2FA?,Yes. Admins on Pro and Enterprise plans can re...,[security-02],True,True,1.230877,354
8,How do I download all data from my workspace?,An admin can request a full workspace export a...,[export-01],True,True,1.814681,397
9,How quickly does support respond to Pro custom...,Pro customers receive a first response within ...,[support-01],True,True,1.284115,343


#### Summary report

In [13]:
report = {
    f"Recall@{TOP_K}": results[f"Recall@{TOP_K}"].mean(),
    f"Precision@{TOP_K}": results[f"Precision@{TOP_K}"].mean(),
    "MRR": results["MRR"].mean(),
    "Faithfulness": results["faithfulness"].mean(),
    "Answer correctness": results["correctness"].mean(),
    "Citation precision": results["citation_precision"].mean(),
    "Citation recall": results["citation_recall"].mean(),
    "Latency p50 [s]": results["latency_s"].median(),
    "Latency p95 [s]": results["latency_s"].quantile(0.95),
    "Tokens / question": results["tokens"].mean(),
    "Tokens total": results["tokens"].sum(),
}
pd.Series(report).round(3).to_frame("value")

,value
Recall@3,0.967
Precision@3,0.356
MRR,0.967
Faithfulness,0.867
Answer correctness,0.800
Citation precision,0.978
Citation recall,0.967
Latency p50 [s],1.330
Latency p95 [s],2.078
Tokens / question,363.933


In [14]:
# Failed or imperfect examples - the most valuable part of the report
failed = results[~results["faithfulness"] | ~results["correctness"] | (results["citation_precision"] < 1)]
print(f"Examples to review: {len(failed)}")
for _, row in failed.iterrows():
    print(f"\nQ: {row['question']}\nA: {row['answer']}\ncited={row['cited']} relevant={row['relevant']}\n{row['judge_notes']}")

Examples to review: 4

Q: What happens to my workspace when the trial ends?
A: When your 21-day Pro trial ends, your workspace is automatically downgraded to the Free plan. Data above the Free plan limits becomes read-only.
cited=['trial-01'] relevant=['trial-01', 'plans-01']
F: Both claims are directly supported by the trial context. | C: The answer describes the downgrade and read-only behavior but omits the Free plan limits of 3 projects and 1 GB.

Q: Which identity providers can I use for SSO?
A: Orbitly SSO supports Okta, Microsoft Entra ID (Azure AD), and Google Workspace. SSO is available on the Enterprise plan.
cited=['sso-01'] relevant=['sso-01']
F: Both the supported identity providers and Enterprise-only availability are stated in the context. | C: The answer lists all three providers and the Enterprise plan requirement, but omits the supported SAML 2.0 and OIDC protocols.

Q: I deleted a project two weeks ago. Can I still restore it?
A: It should still be in the Trash, sinc

### 8. Quality gate
Averages alone don't tell you whether to ship. Agree on thresholds and check them after every change to the pipeline.

When a metric fails, read the failing examples first: sometimes the pipeline is wrong, but sometimes the reference or the judge is too strict (e.g. the reference contains details the question did not ask for). Fix the dataset as carefully as the code.

In [15]:
# Thresholds agreed with the team - a regression test you can run after every change
# (new prompt, model, chunking, embeddings...). In CI, replace print with assert.
THRESHOLDS = {f"Recall@{TOP_K}": 0.9, "Faithfulness": 0.9, "Answer correctness": 0.85,
              "Citation precision": 0.8, "Latency p95 [s]": 10.0}

for name, threshold in THRESHOLDS.items():
    value = report[name]
    ok = value <= threshold if name.startswith("Latency") else value >= threshold
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {value:.3f} (threshold {threshold})")

PASS  Recall@3: 0.967 (threshold 0.9)
FAIL  Faithfulness: 0.867 (threshold 0.9)
FAIL  Answer correctness: 0.800 (threshold 0.85)
PASS  Citation precision: 0.978 (threshold 0.8)
PASS  Latency p95 [s]: 2.078 (threshold 10.0)


### Summary and next steps
- Evaluate **retrieval and generation separately** – it tells you *what* to fix: chunking/embeddings/K, or the prompt/model.
- Keep the dataset in the repo and **extend it with every bug** reported by users.
- LLM judges are not perfect – review a sample of their verdicts by hand from time to time.
- To track experiments over time, upload the dataset to LangSmith and run the same evaluators with `langsmith.evaluate` (see notebook 6_3). Libraries such as RAGAS offer ready-made versions of these metrics.